# 06 — xc 融合漏斗分析（响应 × 资质）

用训好的响应+资质两个模型对全量人群打分，交互式查看**分析漏斗**（全流程 `is_reg → is_finish_task → 授信`，授信口径随所选资质版本：V1 `is_credit_succ` / V2 `is_credit_1v1`）各阶段及端到端综合提升。

- 静态验收报告由 `scripts/run_funnel_eval.py` 产出（runbook 第 6 步）；本 notebook 用于更细的探索：**连续 top-K 提升曲线**（脚本只有离散档位）、**alpha 敏感性曲线**、漏斗对比图、conditional 分段转化、价值捕获曲线、资质 V1/V2 对比。
- 融合口径与脚本/serving 契约一致：`fused_alpha = calib_resp^alpha × calib_qual^(1-alpha)`，alpha 在先于 OOT 评估窗的 fit 窗口（两模型 valid 期重叠段）上拟合，OOT 评估保持诚实；可选 e2e 端到端单模型基线对比。
- 先决条件：已跑 `build_xc_dataset.py`（有 `data/xc_full.csv`，含衍生标签 `is_credit_1v1`）且两个模型已完成 Stage-2 训练。
- 内核需带 xgboost（本机用 conda `env_ml` 环境）。

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../scripts')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from predict_template import Predictor
from run_funnel_eval import FUNNEL_STAGES, RAW_TIER_COLUMN, _model_boundaries, _parse_tier_values
from wdm.config import load_config
from wdm.model.fusion import fit_alpha, fuse
from wdm.utils.paths import model_run_dir

# ---- 参数：选定资质版本与 run-id ----
# 资质 V1（is_credit_succ）：QUAL_PRODUCT='xc_qual_finish',     QUAL_STAGE='is_credit_succ'
# 资质 V2（credit_1v1）：   QUAL_PRODUCT='xc_qual_finish_1v1', QUAL_STAGE='is_credit_1v1'
RESP_PRODUCT, RESP_RUN_ID = 'xc_resp_finish', 'prod_20260610'
QUAL_PRODUCT, QUAL_RUN_ID = 'xc_qual_finish', 'prod_20260610'
QUAL_STAGE = 'is_credit_succ'
E2E_PRODUCT, E2E_RUN_ID = None, None  # 端到端单模型基线，如 'xc_e2e_credit', 'prod_xxx'；None 跳过
ALPHA = None        # None = 在 fit 窗口拟合（与脚本一致）；或手填 [0,1]，0.5 排序上等价朴素乘积
ALPHA_K = 0.10      # alpha 拟合目标 lift@K 的 K（对应脚本 --alpha-k）
TIER_VALUES = None  # 仅 QUAL_STAGE='is_credit_1v1' 时生效，如 '1:120,2:290,3:790'
START_DT = None     # None = 自动取各模型 OOT 边界的较晚者；手填早于 OOT 边界则为 in-sample，结果偏乐观

STAGES = FUNNEL_STAGES[:-1] + [QUAL_STAGE]   # is_reg -> is_finish_task -> 授信

products = {'resp': (RESP_PRODUCT, RESP_RUN_ID), 'qual': (QUAL_PRODUCT, QUAL_RUN_ID)}
if E2E_PRODUCT:
    products['e2e'] = (E2E_PRODUCT, E2E_RUN_ID)
cfgs, preds, bounds = {}, {}, {}
for name, (prod, run_id) in products.items():
    cfgs[name] = load_config(prod)
    bundle = model_run_dir(cfgs[name], run_id)
    preds[name] = Predictor(bundle)
    # 优先读 bundle manifest 里训练时落盘的边界；旧 bundle 回退用当前 CSV 重推（有漂移风险）
    bounds[name] = _model_boundaries(prod, cfgs[name], bundle)
resp_cfg, qual_cfg = cfgs['resp'], cfgs['qual']

# ---- 评估窗（OOT）与 alpha 拟合窗（valid 重叠段，先于评估窗，保证 OOT 评估诚实）----
oot_mins = {n: b['oot_min_dt'] if b else None for n, b in bounds.items()}
if START_DT is None:
    if any(v is None for v in oot_mins.values()):
        raise ValueError(f'无法确定 OOT 边界 {oot_mins}（非时间切分？），请手填 START_DT')
    START_DT = max(oot_mins.values())
elif any(v is not None and START_DT < v for v in oot_mins.values()):
    print(f'警告：START_DT={START_DT} 早于 OOT 边界 {oot_mins}，评估含训练期样本，lift 偏乐观')

FIT_START = None
if ALPHA is None:
    valid_mins = [b.get('valid_min_dt') if b else None for b in bounds.values()]
    if any(v is None for v in valid_mins) or max(valid_mins) >= START_DT:
        print(f'警告：无可用 alpha 拟合窗（valid 边界 {valid_mins}，评估窗起点 {START_DT}），'
              'fused_alpha 退回 alpha=0.5')
        ALPHA = 0.5
    else:
        FIT_START = max(valid_mins)
print('eval window: dt >=', START_DT,
      '| alpha-fit window:', f'[{FIT_START}, {START_DT})' if FIT_START else f'无（alpha={ALPHA}）',
      '| funnel:', ' -> '.join(STAGES))

In [ ]:
# 加载（fit+eval 窗口）并打分
# 与 run_funnel_eval.py 一致：全部 bundle 都带 calibration.json 时才用校准分融合
# （predict_template 契约：跨模型融合必须用校准分），否则警告并退回原始分
time_col = resp_cfg['data']['time_column']
needed = set(STAGES) | {time_col}
for p in preds.values():
    needed |= set(p.base_features)
tier_values = None
if TIER_VALUES:
    assert QUAL_STAGE == 'is_credit_1v1', 'TIER_VALUES 仅适用于资质 V2（is_credit_1v1）'
    tier_values = _parse_tier_values(TIER_VALUES)
    needed |= {RAW_TIER_COLUMN}
df = pd.read_csv(Path(resp_cfg['_repo_root']) / 'data/xc_full.csv', usecols=sorted(needed))

dt_vals = pd.to_numeric(df[time_col], errors='coerce')
load_start = FIT_START if FIT_START is not None else START_DT
keep = dt_vals >= load_start
df = df[keep].reset_index(drop=True)
dt_vals = dt_vals[keep].reset_index(drop=True)
eval_mask = (dt_vals >= START_DT).values
fit_mask = ~eval_mask   # 加载起点即 FIT_START，评估窗之外都是拟合窗

use_calibrated = all(p.has_calibration for p in preds.values())
if not use_calibrated:
    missing = [n for n, p in preds.items() if not p.has_calibration]
    print(f'警告：bundle {missing} 无 calibration.json，用原始分融合（与静态报告口径可能不一致）')
cal = {}
for name, p in preds.items():
    raw = np.asarray(p.predict_proba(df[p.base_features]), dtype=float)
    cal[name] = p.calibrate(raw) if use_calibrated else raw

flags = pd.DataFrame({s: (pd.to_numeric(df[s], errors='coerce').fillna(0) > 0).astype(int)
                      for s in STAGES})
value_vec = None
if tier_values is not None:
    value_vec = pd.to_numeric(df[RAW_TIER_COLUMN], errors='coerce')\
                  .map(tier_values).fillna(0.0).values.astype(float)
print('scores:', 'calibrated' if use_calibrated else 'RAW',
      '| rows: eval', int(eval_mask.sum()), '/ fit', int(fit_mask.sum()),
      '| base rates(eval):', flags[eval_mask].mean().round(4).to_dict())

In [ ]:
# alpha 拟合（与脚本一致：在 fit 窗口上最大化授信段 lift@K），并构建评估窗上的各排序
# fused_alpha 为上线口径（fusion_spec.json 的 serving_formula）；fused 朴素乘积仅作 legacy 对照
alpha_curve = []
if ALPHA is None:
    ALPHA, alpha_source, alpha_curve = fit_alpha(
        cal['resp'][fit_mask], cal['qual'][fit_mask],
        flags[QUAL_STAGE].values[fit_mask], k_pct=ALPHA_K)
else:
    alpha_source = 'manual/fallback'
print(f'fusion alpha={ALPHA:.2f} ({alpha_source})')

rankings = {
    'fused_alpha': fuse(cal['resp'][eval_mask], cal['qual'][eval_mask], ALPHA),
    'fused': cal['resp'][eval_mask] * cal['qual'][eval_mask],
    'resp_only': cal['resp'][eval_mask],
    'qual_only': cal['qual'][eval_mask],
}
if 'e2e' in cal:
    rankings['e2e_only'] = cal['e2e'][eval_mask]
flags_eval = flags[eval_mask].reset_index(drop=True)
value_eval = value_vec[eval_mask] if value_vec is not None else None
n_eval = int(eval_mask.sum())

if alpha_curve:
    ac = pd.DataFrame(alpha_curve)
    plt.figure(figsize=(6, 3.5))
    plt.plot(ac['alpha'], ac['lift_at_k'], marker='o', ms=3)
    plt.axvline(ALPHA, color='red', lw=0.8, ls='--', label=f'alpha={ALPHA:.2f}')
    plt.xlabel('alpha (resp weight)')
    plt.ylabel(f'lift@{ALPHA_K:.0%} on {QUAL_STAGE}')
    plt.title('alpha sensitivity (fit window)')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()

In [ ]:
# 连续 top-K 提升曲线：每个漏斗阶段一张；fused_alpha 为上线口径，应不低于单模型曲线
ks = np.arange(0.01, 0.51, 0.01)
fig, axes = plt.subplots(1, len(STAGES), figsize=(5 * len(STAGES), 4), sharex=True)
for ax, stage in zip(axes, STAGES):
    base = flags_eval[stage].mean()
    for name, sc in rankings.items():
        order = np.argsort(-sc, kind='stable')
        cum_pos = np.cumsum(flags_eval[stage].values[order])
        lifts = []
        for k in ks:
            n_top = max(1, int(np.ceil(k * n_eval)))
            lifts.append(cum_pos[n_top - 1] / n_top / base if base > 0 else np.nan)
        ax.plot(ks, lifts, label=name)
    ax.axhline(1.0, color='gray', lw=0.8, ls='--')
    ax.set_title(stage + (' (end-to-end)' if stage == STAGES[-1] else ''))
    ax.set_xlabel('top-K fraction')
axes[0].set_ylabel('lift vs population')
axes[0].legend()
plt.tight_layout()

In [ ]:
# 选定投放档位 K：全量 vs fused_alpha（上线口径）top-K 的全流程漏斗对比
K = 0.10
n_top = max(1, int(np.ceil(K * n_eval)))
top_idx = np.argsort(-rankings['fused_alpha'], kind='stable')[:n_top]
rows = []
for stage in STAGES:
    rows.append({'stage': stage,
                 'population': flags_eval[stage].mean(),
                 'fused_alpha_topK': flags_eval[stage].values[top_idx].mean()})
cmp_df = pd.DataFrame(rows).set_index('stage')
cmp_df['lift'] = cmp_df['fused_alpha_topK'] / cmp_df['population']
display(cmp_df.round(4))
cmp_df[['population', 'fused_alpha_topK']].plot(
    kind='bar', figsize=(7, 4), rot=0,
    title='Funnel: population vs fused_alpha top-{0:.0%}'.format(K))
plt.ylabel('cumulative rate')
plt.tight_layout()

In [ ]:
# conditional 口径：top-K 内逐段转化率 vs 全量逐段转化率（沿用上格的 K/n_top）
# 用于定位提升来源：is_finish_task 行 lift 高 = 响应段筛人有效；授信行 lift 高 = 资质分把住了授信
def step_conversions(mask):
    prev = mask
    out = {}
    for stage in STAGES:
        flag = flags_eval[stage].values.astype(bool)
        out[stage] = flag[prev].mean() if prev.any() else np.nan
        prev = prev & flag
    return out

cond = {'population': step_conversions(np.ones(n_eval, dtype=bool))}
for name, sc in rankings.items():
    top = np.zeros(n_eval, dtype=bool)
    top[np.argsort(-sc, kind='stable')[:n_top]] = True
    cond[name] = step_conversions(top)
cond_df = pd.DataFrame(cond)
print('top-{0:.0%} 内逐段转化率：'.format(K))
display(cond_df.round(4))
print('相对全量的分段 lift：')
display(cond_df.drop(columns='population').div(cond_df['population'], axis=0).round(3))

In [ ]:
# 价值捕获曲线（仅资质 V2 + TIER_VALUES）：top-K 人均授信价值 vs 全量人均价值
# 对应脚本 --tier-values 的 value_capture 行，这里给出连续 K 曲线
if value_eval is None:
    print('未设置 TIER_VALUES（或资质口径非 is_credit_1v1），跳过价值口径')
else:
    base_v = value_eval.mean()
    plt.figure(figsize=(6, 4))
    for name, sc in rankings.items():
        order = np.argsort(-sc, kind='stable')
        cum_v = np.cumsum(value_eval[order])
        lifts = []
        for k in ks:
            n_k = max(1, int(np.ceil(k * n_eval)))
            lifts.append(cum_v[n_k - 1] / n_k / base_v if base_v > 0 else np.nan)
        plt.plot(ks, lifts, label=name)
    plt.axhline(1.0, color='gray', lw=0.8, ls='--')
    plt.xlabel('top-K fraction')
    plt.ylabel('value lift vs population')
    plt.title('Value capture (mean {0} tier value)'.format(RAW_TIER_COLUMN))
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()

In [ ]:
# 资质 V1 vs V2 对比：汇总已有的 run_funnel_eval.py 静态报告（综合提升行 + 融合 spec）
# 注意：两份报告的 alpha 不同时，对比 fused_alpha 行要连同 alpha 一起看
root = Path(resp_cfg['_repo_root']) / 'artifacts' / 'funnel_eval'
found = False
if root.is_dir():
    for d in sorted(root.iterdir()):
        f = d / 'funnel_eval.csv'
        if not f.is_file():
            continue
        found = True
        print(d.name)
        spec_path = d / 'fusion_spec.json'
        if spec_path.is_file():
            spec = json.loads(spec_path.read_text(encoding='utf-8'))
            print('  alpha={0} ({1}) | calibrated={2} | fit n={3} | eval n={4}'.format(
                spec.get('alpha'), spec.get('alpha_source'), spec.get('calibrated'),
                spec.get('fit_window', {}).get('n_rows'),
                spec.get('eval_window', {}).get('n_rows')))
        r = pd.read_csv(f)
        r = r[(r['view'] == 'absolute') & (r['score'].isin(['fused', 'fused_alpha']))
              & (r['stage'].isin(['is_credit_succ', 'is_credit_1v1']))]
        display(r[['score', 'stage', 'top_k_pct', 'base_rate', 'topk_rate', 'lift']].round(4))
if not found:
    print('尚无 run_funnel_eval.py 产出；先按 docs/PRODUCTION_RUNBOOK.md 第 6 步生成')

## 解读要点

- **`fused_alpha` 是上线口径**（与 `fusion_spec.json` 的 serving_formula 一致）；`fused` 朴素乘积在排序上等价于 alpha=0.5，仅作 legacy 对照。
- 末段授信（V1 `is_credit_succ` / V2 `is_credit_1v1`）的 absolute lift = **端到端综合提升**，是验收主指标；曲线可直接读出任意投放比例下的提升。
- fused_alpha 曲线应在大部分 K 上不低于 resp_only / qual_only（以及 e2e_only，若设置了 E2E_PRODUCT），否则融合无增益（检查资质模型质量、alpha 拟合窗样本量，或换资质口径）。
- alpha 敏感性曲线平坦说明融合对 alpha 不敏感、取 0.5 即可；明显偏向一侧说明该段模型主导排序，此时留意拟合窗样本量（`fit_alpha` 有 min_rows/min_pos 保护，样本不足会回退 0.5）。
- 提升来源定位看 conditional 分段转化表：`is_finish_task` 行 lift 高 = 响应段筛人有效；授信行 lift 高 = 资质分把住了授信。
- 资质口径选型：把 QUAL_PRODUCT/QUAL_STAGE 切到另一版本重跑本 notebook，或对比静态报告汇总格中 V1/V2 的综合提升（连同各自 alpha），取贴合投放目标且提升更高者上线；V2 配合 TIER_VALUES 补看人均价值口径。